In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import date, timedelta
from typing import Optional

from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

In [ ]:
# Ого, данных действительно много
data = pl.read_parquet('data/train.parquet')
print(data.shape)

In [ ]:
# Базовые проверки корректности
print(data.dtypes)
print(data.null_count()) 

In [ ]:
# Бегло посмотрим на сами данные
print(data["event_date"].min(), "по", data["event_date"].max())
data.head(5)

### Что происходит с общей временной динамикой данных?

Давайте агрегируем несколько глобальных дневных статистик и построим графики.

In [ ]:
daily_agg = (
    data.group_by("event_date")
    .agg(
        pl.len().alias("n_users"),
        pl.sum("gmv").alias("gmv_sum"),
        pl.sum("to_cart").alias("to_cart_sum"),
        pl.sum("to_ord").alias("to_ord_sum"),
        pl.sum("searches").alias("searches_sum"),
    )
    .sort("event_date")
)

daily_agg.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Активные пользователи
axes[0].plot(daily_agg["event_date"], daily_agg["n_users"], marker=".", linestyle="-", color="steelblue")
axes[0].set_ylabel("Число пользователей")
axes[0].set_title("Ежедневное число активных пользователей")
axes[0].grid(True, alpha=0.3)

# Суммарный GMV
axes[1].plot(daily_agg["event_date"], daily_agg["gmv_sum"], marker=".", linestyle="-", color="darkorange")
axes[1].set_ylabel("Суммарный GMV")
axes[1].set_title("Ежедневный GMV")
axes[1].grid(True, alpha=0.3)

plt.xlabel("Дата")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(daily_agg["event_date"], daily_agg["searches_sum"], label="Поисковые запросы", color="purple")
plt.plot(daily_agg["event_date"], daily_agg["to_cart_sum"], label="Добавления в корзину", color="green")
plt.plot(daily_agg["event_date"], daily_agg["to_ord_sum"], label="Заказы", color="red")
plt.legend()
plt.title("Ежедневные поиски, добавления в корзину и заказы")
plt.ylabel("Количество")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Подготовим признаки 

- Не у всех есть компьютеры с большим объёмом оперативной памяти, особенно в середине 2026 года
- Поэтому сначала построим временные признаки и целевые переменные...
- И будем рассчитывать их пакетами пользователей
- После этого можно будет работать с извлечёнными данными как с существенно более простой задачей регрессии

In [ ]:
# Дополнительные признаки оставим как домашнее задание
DEFAULT_AGGS = ["sum", "max", "std", "mean"] # Функции агрегации
DEFAULT_VALUE_COLS = ["gmv", "searches"] # Используемые переменные для агрегации 

# И зададим место, куда будем сохранять результаты
FEATURES_DIR = "data/v2/features"

# Пока выберем простой вариант
N_FOLDS = 4

# Учитывайте объём своей оперативной памяти
# На компьютерах с очень небольшой памятью учитывайте, что данные отсортированы по пользователю, а далее — по дате события
BATCH_SIZE = 50_000

# окно = [якорная дата - начало смещения, якорная дата - конец смещения] (обе границы включаются)
DEFAULT_WINDOWS = [
    ("30d", 29, 0),    # [t-29, t-0]  (включая якорную дату)
    ("60d", 59, 30),   # [t-59, t-30]
    ("90d", 89, 60),   # [t-89, t-60]
]

output_dir = Path(FEATURES_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Построим ключевые якорные даты для временных окон кросс-валидации
def generate_cv_anchor_dates(
    data: pl.DataFrame,
    prediction_horizon_days: int = 30,
    stride_days: int = 14,
    min_history_days: int = 90,
    n_folds: Optional[int] = None,
) -> list[date]:
   
    min_date = data["event_date"].min()
    max_date = data["event_date"].max()

    latest_anchor = max_date - timedelta(days = prediction_horizon_days)
    earliest_anchor = min_date + timedelta(days = min_history_days - 1)

    # Проверка корректности
    if latest_anchor < earliest_anchor:
        raise ValueError(
            f"Недостаточно данных: требуется не менее {min_history_days + prediction_horizon_days - 1} дней, "
            f"получено {(max_date - min_date).days}."
        )

    n_steps = (latest_anchor - earliest_anchor).days // stride_days
    all_anchors = [latest_anchor - timedelta(days=i * stride_days) for i in range(n_steps + 1)]
    all_anchors = sorted(all_anchors)

    if n_folds is not None:
        if n_folds <= 0:
            raise ValueError("n_folds должно быть положительным целым числом.")
        return all_anchors[-n_folds:]
    
    return all_anchors

In [ ]:
# Вспомогательные "рецепты" признаков
# Добавьте собственные признаки; также можно рассмотреть путь токенизации поведения :)
def _window_agg_exprs(
    anchor_val: date,
    windows: list[tuple[str, int, int]],
    value_cols: list[str],
    aggs: list[str],
) -> list[pl.Expr]:
    exprs = []
    for w_name, start_off, end_off in windows:
        w_start = anchor_val - timedelta(days = start_off)
        w_end = anchor_val - timedelta(days = end_off)
        mask = pl.col("event_date").is_between(w_start, w_end)

        for col in value_cols:
            for agg in aggs:
                if agg == "sum":
                    e = pl.when(mask).then(pl.col(col)).otherwise(0.0).sum()
                elif agg == "max":
                    e = pl.when(mask).then(pl.col(col)).otherwise(None).max()
                elif agg == "std":
                    e = pl.when(mask).then(pl.col(col)).otherwise(None).std()
                elif agg == "mean":
                    e = pl.when(mask).then(pl.col(col)).otherwise(None).mean()
                elif agg == "count":
                    e = pl.when(mask).then(1).otherwise(0).sum()
                else:
                    raise ValueError(f"Неизвестная агрегация: {agg}")
                exprs.append(e.alias(f"{col}_{agg}_{w_name}"))
    return exprs

In [ ]:
# Подготовка признаков, пригодная для пакетной обработки
# Можно извлечь все признаки сразу, однако это, вероятно, потребует более 40 ГБ оперативной памяти
def generate_features(
    data: pl.DataFrame,
    anchor_dates: list[date],
    user_ids: Optional[list[int]] = None,
    value_cols: list[str] = DEFAULT_VALUE_COLS,
    windows: list[tuple[str, int, int]] = DEFAULT_WINDOWS,
    aggs: list[str] = DEFAULT_AGGS,
) -> pl.DataFrame:
    if user_ids is None:
        user_ids = data["user_id"].unique().sort().to_list()

    max_back = max(w[1] for w in windows)
    min_anchor, max_anchor = min(anchor_dates), max(anchor_dates)

    data_f = data.filter(
        pl.col("user_id").is_in(user_ids)
        & pl.col("event_date").is_between(
            min_anchor - timedelta(days=max_back),
            max_anchor,
        )
    )

    parts = []
    for a in anchor_dates:
        ad = data_f.filter(pl.col("event_date") >= a - timedelta(days=max_back))
        if len(ad) > 0:
            features = (
                ad.group_by("user_id")
                  .agg(_window_agg_exprs(a, windows, value_cols, aggs))
                  .with_columns(anchor_date=pl.lit(a))
            )
            parts.append(features)

    features_df = pl.concat(parts, how="diagonal_relaxed") if parts else pl.DataFrame()
    index_df = (
        pl.DataFrame({"anchor_date": anchor_dates})
        .join(pl.DataFrame({"user_id": user_ids}), how="cross")
    )
    result = index_df.join(features_df, on=["anchor_date", "user_id"], how="left")

    feature_cols = [c for c in result.columns if c not in ["anchor_date", "user_id"]]
    fill_exprs = [pl.col(c).fill_null(0.0) for c in feature_cols]

    
    return result.with_columns(fill_exprs)


In [ ]:
# Построение целевой переменной, пригодное для пакетной обработки
def generate_targets(
    data: pl.DataFrame,
    anchor_dates: list[date],
    user_ids: Optional[list[int]] = None,
    horizon_days: int = 30,
    target_col: str = "gmv",
) -> pl.DataFrame:
    if user_ids is None:
        user_ids = data["user_id"].unique().sort().to_list()

    index_df = (
        pl.DataFrame({"anchor_date": anchor_dates})
        .join(pl.DataFrame({"user_id": user_ids}), how="cross")
    )

    parts = []
    for a in anchor_dates:
        t_start = a + timedelta(days=1)
        t_end   = a + timedelta(days=horizon_days)
        tgt = (
            data.filter(
                pl.col("user_id").is_in(user_ids)
                & pl.col("event_date").is_between(t_start, t_end)
            )
            .group_by("user_id")
            .agg(pl.col(target_col).sum().alias("target"))
            .with_columns(anchor_date=pl.lit(a))
        )
        parts.append(tgt)

    tgt_df = pl.concat(parts, how="diagonal_relaxed") if parts else pl.DataFrame()
    targets = index_df.join(tgt_df, on=["anchor_date", "user_id"], how="left")
    return targets.with_columns(pl.col("target").fill_null(0.0))

In [ ]:
# Подготовим структуру временных якорей
anchors_time_folds = generate_cv_anchor_dates(data, n_folds = N_FOLDS)
anchor_end_of_time = data["event_date"].max()

print(f"Фолды временной кросс-валидации: {len(anchors_time_folds)} | {anchors_time_folds[0]} → "
    f"{anchors_time_folds[-1]} | шаг = 14 дней")
print(f"Срез для прогноза: {anchor_end_of_time} → прогноз [{anchor_end_of_time + timedelta(days = 1)}, "
    f"{anchor_end_of_time + timedelta(days = 30)}]")

In [ ]:
# Подготовимся к пакетному извлечению
user_ids = data["user_id"].unique().sort().to_list()

n_users   = len(user_ids)
n_batches = (n_users + BATCH_SIZE - 1) // BATCH_SIZE
print(f"\n{n_users} пользователей → {n_batches} пакетов на фолд (размер пакета = {BATCH_SIZE})\n")

### Запускаем расчёт

In [ ]:
# 1. Обрабатываем фолды кросс-валидации 

for fold_idx, anchor in enumerate(anchors_time_folds):

    fold_name = f"fold_{fold_idx:02d}"
    fold_dir = output_dir / fold_name
    fold_dir.mkdir(parents=True, exist_ok=True)

    for batch in range(n_batches):
        current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
        features = generate_features(data, [anchor], user_ids = current_batch)
        targets = generate_targets(data, [anchor], user_ids = current_batch,
                                horizon_days = 30)
            
        out_df = features.join(targets, on=["anchor_date", "user_id"], how="left")
        out_df = out_df.with_columns(pl.col("target").fill_null(0.0))
        out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")
            
    print(f"  Обработан {fold_name} (якорная дата: {anchor})")

In [ ]:
# 2. Обрабатываем финальный срез для прогноза

fold_dir = output_dir / "fold_end"
fold_dir.mkdir(parents=True, exist_ok=True)

for batch in range(n_batches):
    current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
    feats = generate_features(data, [anchor_end_of_time], user_ids = current_batch)
    out_df = feats.with_columns(pl.lit(None).cast(pl.Float64).alias("target"))
    out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")
        
print(f"  Обработан последний фолд (якорная дата: {anchor_end_of_time})")

### Теперь перейдём к машинному обучению

- Посмотрим, с чем мы имеем дело
- После этого определим следующий лучший план действий

In [ ]:
# Безопасная реализация RMSLE с log1p и отсечением недопустимых значений
def rmsle(y_true, y_pred):
    log_true = np.log1p(np.clip(y_true, 0, None))
    log_pred = np.log1p(np.clip(y_pred, 0, None))
    return np.sqrt(mean_squared_error(log_true, log_pred))

# Небольшая вспомогательная функция для чтения подготовленных фолдов признаков
def read_fold(output_dir: str | Path, fold_name: str):
    pattern = str(Path(output_dir) / fold_name / "batch_*.parquet")
    return pl.read_parquet(pattern)

In [ ]:
folds = []
for fold_idx, anchor in enumerate(anchors_time_folds):
    fold_name = f"fold_{fold_idx:02d}"
    fold_df = read_fold(FEATURES_DIR, fold_name)
    folds.append(fold_df)

### Может быть, немного CatBoost?

- Возможно, с распределением Tweedie из-за большого количества нулевых значений?
- Возможно, стоит добавить признаки: можно предположить, что gmv_sum_30/60/90 — самые важные?
- ...
- Нет, давайте пока используем простую авторегрессию

In [ ]:
# Это последний обучающий временной срез с честно доступной целевой переменной 
folds[3].head(5)

In [ ]:
# Используем эту целевую переменную как основу для сабмита
submit_naive = folds[3].select(["user_id", "target"])
submit_naive.columns = ["user_id", "predict"]
print(submit_naive.shape)

In [ ]:
# А теперь быстро сохраним его ещё до какого-либо ML и посмотрим себя в лидерборде!
submit_naive.write_csv("data/sample_naive_submit.csv")